In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

In [4]:
device

device(type='cuda')

In [5]:
df = pd.read_csv('fmnist_small.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [6]:
x=df.iloc[:,1:].values
y = df.iloc[:,0].values

In [11]:
y.shape

(6000,)

In [7]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=43)

In [8]:
x_train.shape

(4800, 784)

In [9]:
x_test.shape

(1200, 784)

In [12]:
y.shape

(6000,)

In [13]:
x_train= x_train/255.0
x_test = x_test/255.0

In [14]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features,dtype=torch.float32).reshape(-1,1,28,28)
        self.labels = torch.tensor(labels,dtype=torch.long)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index],self.labels[index]

In [15]:
train_data = CustomDataset(x_train,y_train)
test_data = CustomDataset(x_test,y_test)

In [16]:
train_loader = DataLoader(train_data,batch_size=32,shuffle=True)
test_data = DataLoader(test_data,batch_size=43,shuffle=False)

In [25]:
class MyCNN(nn.Module):
    def __init__(self,input_dim):
        super().__init__()

        self.linear = nn.Sequential(
            nn.Conv2d(input_dim,32,kernel_size=2,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2,stride=2),

            nn.Conv2d(32,64,kernel_size=2,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2,stride=2)
            
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(64,10)
        )

    def forward(self,x):
        out = self.linear(x)
        result = self.classifier(out)
        return result

In [26]:
lr = 0.1
epochs =100
input_dim = 784

In [31]:
Model = MyCNN(1)
model = Model.to(device)
loss_function=nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(),lr =lr,weight_decay=1e-4)

In [32]:
for epoch in range(epochs):
    for batch_features,batch_labels in train_loader:
        
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        output = model(batch_features)

        loss = loss_function(output,batch_labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        print(f"loss {loss}")

loss 2.3747661113739014
loss 2.2908315658569336
loss 2.1694459915161133
loss 2.043334484100342
loss 2.0028209686279297
loss 2.0723910331726074
loss 1.9040815830230713
loss 1.7607803344726562
loss 1.6937628984451294
loss 1.5019590854644775
loss 1.3237054347991943
loss 1.8279826641082764
loss 1.5854355096817017
loss 1.346168041229248
loss 1.5253759622573853
loss 1.3646061420440674
loss 1.884007453918457
loss 1.4823349714279175
loss 1.5022088289260864
loss 1.1014378070831299
loss 0.9741162061691284
loss 0.9736294746398926
loss 1.5832746028900146
loss 1.346404790878296
loss 0.9967961311340332
loss 0.9931407570838928
loss 1.8332979679107666
loss 1.1954084634780884
loss 0.8846943974494934
loss 1.3661201000213623
loss 1.3545689582824707
loss 1.1584428548812866
loss 0.7438844442367554
loss 0.9599952697753906
loss 1.4126415252685547
loss 1.2496747970581055
loss 1.5073119401931763
loss 0.9230213761329651
loss 1.4762104749679565
loss 1.0870922803878784
loss 1.2353782653808594
loss 1.2217018604278